# Music Transformer & Anticipatory Music Transformer

**CS 89.02 / MUS 14.05 — Music and AI — Week 5**

The **Music Transformer** (Huang & Eck, 2018) was one of the first models to
demonstrate that self-attention could generate piano music with coherent long-term
structure — repeating themes, tension and resolution, and phrase-level organization
that LSTMs struggled to achieve.

The key innovation was **relative positional encoding**: rather than encoding absolute
position ("this is note 47 in the sequence"), the model encodes the *distance* between
notes ("these two notes are 8 steps apart"). This makes the model better at learning
patterns like repetition and transposition that depend on relative, not absolute, position.

The **Anticipatory Music Transformer** (Thickstun et al., 2023) extends this idea
with an interleaved tokenization scheme that separates melody from accompaniment,
enabling the model to generate accompaniment conditioned on a given melody (infilling).

In this notebook, we will:
1. Load a pre-trained music transformer model
2. Generate music unconditionally
3. Generate accompaniment for a given melody
4. Visualize attention patterns to understand what the model has learned

In [ ]:
# Install dependencies
# The Anticipatory Music Transformer (AMT) can be installed from its repository.
# If installation fails on Colab, we provide a fallback using note-seq and a
# basic transformer.

!pip install -q pretty_midi matplotlib numpy torch

# Try installing AMT
try:
    !pip install -q anticipatory-music-transformer 2>/dev/null
    AMT_AVAILABLE = True
except:
    AMT_AVAILABLE = False

# Fallback: clone the repo if pip install is not available
import os
if not AMT_AVAILABLE:
    if not os.path.exists('anticipatory-music-transformer'):
        !git clone https://github.com/jthickstun/anticipatory-music-transformer.git 2>/dev/null || true
        !cd anticipatory-music-transformer && pip install -q -e . 2>/dev/null || true
    try:
        import anticipatory_music_transformer
        AMT_AVAILABLE = True
    except:
        print("AMT not available. We will use a simplified demonstration.")
        AMT_AVAILABLE = False

print(f"AMT available: {AMT_AVAILABLE}")

## The Transformer Architecture for Music

### Self-Attention

The core operation of the transformer is **self-attention**. For each token in the
sequence, the model computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

where Q (query), K (key), and V (value) are linear projections of the input.

Musically, this means: when generating the next note, the model can directly attend
to *any* previous note — not just the most recent ones (as in Markov models) or
whatever the hidden state has managed to compress (as in LSTMs).

### Relative Positional Encoding

Standard transformers use absolute positional encodings: each position gets a fixed
vector. The Music Transformer instead uses **relative** positional encodings, where
the attention score between positions i and j depends on their *distance* (i - j).

This is important for music because:
- A melody repeated at a different point in time should be recognized as the same pattern
- Harmonic intervals depend on pitch *distance*, not absolute pitch
- Rhythmic patterns are about relative timing

### Anticipatory Music Transformer

The AMT uses an **interleaved tokenization** scheme:
- Time advances in discrete steps
- At each time step, tokens alternate between "anticipated" (future) and "current" events
- This allows the model to see the melody ahead and generate appropriate accompaniment
- Enables **infilling**: given the melody track, generate the harmony/bass tracks

In [ ]:
import torch
import numpy as np
import pretty_midi
import matplotlib.pyplot as plt
from IPython.display import Audio, display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if AMT_AVAILABLE:
    # Load the pre-trained AMT model
    # The model checkpoint will be downloaded automatically
    try:
        from anticipatory_music_transformer import AMTModel, AMTTokenizer
        model = AMTModel.from_pretrained('jthickstun/amt-base').to(device)
        tokenizer = AMTTokenizer()
        model.eval()
        print("AMT model loaded successfully.")
        print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    except Exception as e:
        print(f"Could not load AMT model: {e}")
        print("Falling back to demonstration mode.")
        AMT_AVAILABLE = False
else:
    print("Running in demonstration mode (no AMT model).")
    print("To use the full AMT, install from: https://github.com/jthickstun/anticipatory-music-transformer")

## Unconditional Generation

First, let us generate a piece from scratch. The model starts with a minimal prompt
(e.g., a time signature and tempo token) and autoregressively generates the rest.

In [ ]:
import os
os.makedirs('output', exist_ok=True)

if AMT_AVAILABLE:
    # Unconditional generation with AMT
    print("Generating a piece from scratch...")

    with torch.no_grad():
        # Generate token sequence
        tokens = model.generate(
            max_length=512,
            temperature=0.95,
            top_p=0.95
        )

    # Decode tokens to MIDI
    midi = tokenizer.decode(tokens)
    midi.write('output/amt_unconditional.mid')
    print("Saved to output/amt_unconditional.mid")

    # Synthesize and play
    try:
        audio = midi.fluidsynth(fs=22050)
        display(Audio(audio, rate=22050))
    except:
        print("FluidSynth not available. Download the MIDI file to listen.")

else:
    # Demonstration: create a simple example showing what AMT output looks like
    print("=== Demonstration Mode ===")
    print("The AMT model generates multi-track MIDI with melody and accompaniment.")
    print("Here is an example of what the token stream looks like:\n")

    example_tokens = [
        "TIME_SHIFT_0.125",
        "NOTE_ON_60",      # C4 - melody
        "NOTE_ON_48",      # C3 - bass
        "NOTE_ON_52",      # E3 - harmony
        "NOTE_ON_55",      # G3 - harmony
        "TIME_SHIFT_0.5",
        "NOTE_OFF_60",
        "NOTE_ON_62",      # D4 - melody
        "TIME_SHIFT_0.25",
        "NOTE_OFF_48",
        "NOTE_OFF_52",
        "NOTE_OFF_55",
        "NOTE_ON_50",      # D3 - bass
        "NOTE_ON_53",      # F3 - harmony
        "NOTE_ON_57",      # A3 - harmony
    ]

    for tok in example_tokens:
        print(f"  {tok}")

    print("\nThe key insight: the model learns to coordinate melody, harmony, and bass")
    print("into coherent musical progressions across hundreds of time steps.")

    # Create a simple example MIDI to demonstrate the concept
    midi = pretty_midi.PrettyMIDI(initial_tempo=100)

    # Melody
    melody_inst = pretty_midi.Instrument(program=0, name='Melody')
    melody_notes = [72, 74, 76, 77, 79, 77, 76, 74, 72, 71, 72, 74, 72, 71, 69, 67]
    for i, pitch in enumerate(melody_notes):
        melody_inst.notes.append(pretty_midi.Note(
            velocity=90, pitch=pitch, start=i*0.5, end=(i+1)*0.5 - 0.05))
    midi.instruments.append(melody_inst)

    # Accompaniment (block chords)
    accomp_inst = pretty_midi.Instrument(program=0, name='Accompaniment')
    chords = [
        ([48, 52, 55], 0, 2),    # C major
        ([50, 53, 57], 2, 4),    # Dm
        ([48, 52, 55], 4, 6),    # C major
        ([47, 50, 55], 6, 8),    # G/B
    ]
    for pitches, start, end in chords:
        for p in pitches:
            accomp_inst.notes.append(pretty_midi.Note(
                velocity=60, pitch=p, start=start, end=end - 0.05))
    midi.instruments.append(accomp_inst)

    midi.write('output/amt_demo_unconditional.mid')
    print("\nSaved demo MIDI to output/amt_demo_unconditional.mid")

## Infilling / Accompaniment

One of the most powerful capabilities of the Anticipatory Music Transformer is
**infilling**: given a melody, the model generates an appropriate accompaniment.

This is possible because of the interleaved tokenization: the model can see
"anticipated" melody tokens in the future and use them to inform its choices
for the current accompaniment tokens.

This is analogous to a human accompanist who reads ahead in the score.

In [ ]:
# Define a simple melody for accompaniment generation
melody_midi = pretty_midi.PrettyMIDI(initial_tempo=120)
melody_track = pretty_midi.Instrument(program=0, name='Melody')

# "Twinkle Twinkle" melody
twinkle_notes = [
    (60, 0.0, 0.5), (60, 0.5, 1.0), (67, 1.0, 1.5), (67, 1.5, 2.0),
    (69, 2.0, 2.5), (69, 2.5, 3.0), (67, 3.0, 4.0),
    (65, 4.0, 4.5), (65, 4.5, 5.0), (64, 5.0, 5.5), (64, 5.5, 6.0),
    (62, 6.0, 6.5), (62, 6.5, 7.0), (60, 7.0, 8.0),
]

for pitch, start, end in twinkle_notes:
    melody_track.notes.append(pretty_midi.Note(
        velocity=90, pitch=pitch, start=start, end=end - 0.02))
melody_midi.instruments.append(melody_track)
melody_midi.write('output/input_melody.mid')

if AMT_AVAILABLE:
    print("Generating accompaniment for the input melody...")

    # Tokenize the input melody
    melody_tokens = tokenizer.encode(melody_midi, melody_only=True)

    # Generate accompaniment
    with torch.no_grad():
        filled_tokens = model.infill(
            melody_tokens,
            temperature=0.9,
            top_p=0.95
        )

    # Decode to MIDI
    result_midi = tokenizer.decode(filled_tokens)
    result_midi.write('output/amt_accompaniment.mid')
    print("Saved to output/amt_accompaniment.mid")

    # Synthesize and play
    try:
        audio = result_midi.fluidsynth(fs=22050)
        display(Audio(audio, rate=22050))
    except:
        print("FluidSynth not available. Download the MIDI file to listen.")
else:
    print("=== Demonstration: Accompaniment Generation ===")
    print("Given a melody, AMT generates harmonically appropriate accompaniment.\n")
    print("Input:  C  C  G  G  | A  A  G  - |")
    print("                                   ")
    print("AMT might generate:")
    print("Bass:   C  .  .  .  | F  .  C  . |  (root motion)")
    print("Chord:  CEG .  .  . | FAC .  CEG . |  (triads)")
    print("")
    print("The model learns common harmonic progressions from training data")
    print("and applies them in context. It can handle modulations, passing")
    print("tones, and voice leading.")

    # Create a demo with hand-crafted accompaniment
    full_midi = pretty_midi.PrettyMIDI(initial_tempo=120)

    # Copy melody
    mel = pretty_midi.Instrument(program=0, name='Melody')
    for pitch, start, end in twinkle_notes:
        mel.notes.append(pretty_midi.Note(velocity=90, pitch=pitch, start=start, end=end-0.02))
    full_midi.instruments.append(mel)

    # Add accompaniment
    acc = pretty_midi.Instrument(program=0, name='Accompaniment')
    accomp = [
        # C major
        (48, 0.0, 2.0), (52, 0.0, 2.0), (55, 0.0, 2.0),
        # C major (second half)
        (48, 2.0, 4.0), (52, 2.0, 4.0), (55, 2.0, 4.0),
        # F major
        (53, 4.0, 6.0), (48, 4.0, 6.0), (45, 4.0, 6.0),
        # C major
        (48, 6.0, 8.0), (52, 6.0, 8.0), (55, 6.0, 8.0),
    ]
    for pitch, start, end in accomp:
        acc.notes.append(pretty_midi.Note(velocity=60, pitch=pitch, start=start, end=end-0.02))
    full_midi.instruments.append(acc)

    full_midi.write('output/amt_demo_accompaniment.mid')
    print("\nSaved demo with accompaniment to output/amt_demo_accompaniment.mid")

## Analysis: Attention Patterns

One advantage of the transformer architecture is interpretability through
attention weights. We can ask: when generating a particular note, which
previous notes did the model attend to most strongly?

In music, we might expect the model to attend to:
- Notes at the same metric position in previous measures (rhythmic repetition)
- Notes that form harmonic intervals with the current note
- The beginning of the current phrase
- Melodic motifs that are being repeated or varied

Let us visualize these attention patterns.

In [ ]:
if AMT_AVAILABLE:
    # Extract attention weights from the model
    print("Extracting attention weights...")

    # Re-run generation with attention output
    with torch.no_grad():
        tokens_tensor = torch.tensor([tokens[:64]]).to(device)
        outputs = model(
            tokens_tensor,
            output_attentions=True
        )
        attentions = outputs.attentions  # list of (batch, heads, seq, seq)

    # Visualize attention from the last layer
    last_layer_attn = attentions[-1][0].cpu().numpy()  # (heads, seq, seq)

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    for idx, ax in enumerate(axes.flat):
        if idx < last_layer_attn.shape[0]:
            im = ax.imshow(last_layer_attn[idx], aspect='auto', cmap='Blues')
            ax.set_title(f'Head {idx}')
            ax.set_xlabel('Key Position')
            ax.set_ylabel('Query Position')
            plt.colorbar(im, ax=ax, fraction=0.046)

    fig.suptitle('Attention Weights (Last Layer)', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Average attention across heads
    avg_attn = last_layer_attn.mean(axis=0)
    plt.figure(figsize=(10, 8))
    plt.imshow(avg_attn, aspect='auto', cmap='Blues')
    plt.colorbar(label='Attention Weight')
    plt.xlabel('Key Position (attended to)')
    plt.ylabel('Query Position (attending from)')
    plt.title('Average Attention Across All Heads (Last Layer)')
    plt.tight_layout()
    plt.show()

else:
    # Demonstration: show what attention patterns typically look like
    print("=== Demonstration: Attention Pattern Visualization ===")
    print("In a trained Music Transformer, attention heads specialize:\n")

    # Create synthetic attention patterns for demonstration
    seq_len = 32
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Head 0: Local attention (nearby notes)
    local_attn = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        for j in range(seq_len):
            if j <= i:
                local_attn[i, j] = np.exp(-0.5 * (i - j))
    local_attn /= local_attn.sum(axis=1, keepdims=True) + 1e-8
    im = axes[0, 0].imshow(local_attn, aspect='auto', cmap='Blues')
    axes[0, 0].set_title('Head 0: Local Context\n(attends to nearby notes)')
    plt.colorbar(im, ax=axes[0, 0], fraction=0.046)

    # Head 1: Periodic attention (every 8 steps = metric structure)
    periodic_attn = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        for j in range(seq_len):
            if j <= i and (i - j) % 8 == 0:
                periodic_attn[i, j] = 1.0
            elif j <= i:
                periodic_attn[i, j] = 0.05
    periodic_attn /= periodic_attn.sum(axis=1, keepdims=True) + 1e-8
    im = axes[0, 1].imshow(periodic_attn, aspect='auto', cmap='Blues')
    axes[0, 1].set_title('Head 1: Metric Structure\n(attends to same beat position)')
    plt.colorbar(im, ax=axes[0, 1], fraction=0.046)

    # Head 2: Beginning of phrase
    phrase_attn = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        phrase_start = (i // 16) * 16
        for j in range(seq_len):
            if j <= i:
                if j == phrase_start:
                    phrase_attn[i, j] = 1.0
                else:
                    phrase_attn[i, j] = 0.03
    phrase_attn /= phrase_attn.sum(axis=1, keepdims=True) + 1e-8
    im = axes[1, 0].imshow(phrase_attn, aspect='auto', cmap='Blues')
    axes[1, 0].set_title('Head 2: Phrase Boundaries\n(attends to phrase beginnings)')
    plt.colorbar(im, ax=axes[1, 0], fraction=0.046)

    # Head 3: Harmonic intervals (attends to bass notes)
    harmonic_attn = np.zeros((seq_len, seq_len))
    bass_positions = [0, 4, 8, 12, 16, 20, 24, 28]
    for i in range(seq_len):
        for j in range(seq_len):
            if j <= i:
                if j in bass_positions:
                    harmonic_attn[i, j] = 0.8
                else:
                    harmonic_attn[i, j] = 0.05
    harmonic_attn /= harmonic_attn.sum(axis=1, keepdims=True) + 1e-8
    im = axes[1, 1].imshow(harmonic_attn, aspect='auto', cmap='Blues')
    axes[1, 1].set_title('Head 3: Harmonic Reference\n(attends to chord roots)')
    plt.colorbar(im, ax=axes[1, 1], fraction=0.046)

    for ax in axes.flat:
        ax.set_xlabel('Key Position')
        ax.set_ylabel('Query Position')

    fig.suptitle('Typical Attention Head Specialization in Music Transformers', fontsize=14)
    plt.tight_layout()
    plt.show()

    print("\nKey observations from Music Transformer attention analysis:")
    print("- Different heads specialize in different musical relationships")
    print("- Some heads track local context (nearby notes, voice leading)")
    print("- Some heads track metric structure (same beat across measures)")
    print("- Some heads track phrase-level structure (phrase beginnings/endings)")
    print("- Some heads track harmonic relationships (chord tones, bass notes)")
    print("\nThis specialization emerges from training — it is not hard-coded.")